# Trabajo Práctico 2 - Grupo 02

### Modelo Bayes Naive

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo de esta notebook es evaluar Bayes Naive usando **FastText**

FastText representa cada documento como el **promedio de los vectores de sus tokens**, donde cada vector captura similitud semántica y maneja palabras fuera del vocabulario via subpalabras (n-gramas de caracteres).

Se usa `GaussianNB` en lugar de `MultinomialNB` porque los embeddings de FastText contienen valores negativos, lo cual es incompatible con `MultinomialNB`.

## Importación e instalación de dependencias


In [1]:
!pip install -q spacy fasttext-wheel
!python -m spacy download es_core_news_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 90.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV

In [5]:
from common.preprocessing import clean_classical
from common.data_utils    import get_split, SEED
from common.embeddings    import make_fasttext
from common.evaluation    import evaluate
from common.io_utils      import save_model, load_model

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [8]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")

X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [9]:
print("Preprocesando con clean_clasical sin lematizacion")
X_train_clean = np.array([clean_classical(t, lemmatize=False) for t in X_train_raw])
X_val_clean   = np.array([clean_classical(t, lemmatize=False) for t in X_val_raw])
X_test_clean  = np.array([clean_classical(t, lemmatize=False) for t in test['text'].values])
print(f"Train: {len(X_train_clean):,} | Val: {len(X_val_clean):,} | Test: {len(X_test_clean):,}")

Preprocesando con clean_clasical sin lematizacion
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N6: Bayes Naive + FastText

Se usa el modelo preentrenado `cc.es.300` de Meta: entrenado sobre Common Crawl en español, 300 dimensiones.

Los embeddings se calculan **una sola vez** para train, val y test antes de cualquier entrenamiento. Esto porque FastText es un modelo fijo preentrenado: sus pesos no cambian con nuestros datos.

In [10]:
vec = make_fasttext()

X_train_ft = vec.fit_transform(X_train_clean)
X_val_ft   = vec.transform(X_val_clean)
X_test_ft  = vec.transform(X_test_clean)

print(f"Shape train : {X_train_ft.shape}")
print(f"Shape val   : {X_val_ft.shape}")
print(f"Shape test  : {X_test_ft.shape}")

[FastText] Modelo no encontrado en 'cc.es.300.bin'.
[FastText] Descargando cc.es.300.bin (~7 GB)...

[FastText] Cargando modelo desde 'cc.es.300.bin'...
[FastText] Modelo cargado. Dimensión: 300


Shape train : (40800, 300)
Shape val   : (10200, 300)
Shape test  : (8500, 300)


`GaussianNB` extiende Bayes Naive para trabajar con features continuas, asumiendo que cada una sigue una distribución normal por clase.

El único hiperparámetro es `var_smoothing`, que estabiliza el modelo cuando alguna dimensión del embedding tiene varianza cercana a cero.

In [11]:
clf = GaussianNB()
clf.fit(X_train_ft, y_train)
y_pred = clf.predict(X_val_ft)

evaluate(
    "nb_ft_v1", y_val, y_pred,
    hyperparams={"var_smoothing": 1e-9, "vectorizer": "FastText cc.es.300"}
)


=== nb_ft_v1 ===
Hiperparámetros: {'var_smoothing': 1e-09, 'vectorizer': 'FastText cc.es.300'}

F1-macro:  0.5015
Precision: 0.5149
Recall:    0.5045
Accuracy:  0.5381

              precision    recall  f1-score   support

    negativa     0.6787    0.4809    0.5629      4080
      neutra     0.2694    0.3363    0.2992      2040
    positiva     0.5965    0.6963    0.6425      4080

    accuracy                         0.5381     10200
   macro avg     0.5149    0.5045    0.5015     10200
weighted avg     0.5639    0.5381    0.5420     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      1962    1030      1088
neutra         520     686       834
positiva       409     830      2841


## Busqueda de hiperparámetros


Se explora `var_smoothing` en escala logarítmica.

In [15]:
param_grid = {"var_smoothing": np.logspace(-12, -1, 12)}

gs = GridSearchCV(
    GaussianNB(),
    param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=1
)
gs.fit(X_train_ft, y_train)

print(f"Mejor var_smoothing : {gs.best_params_['var_smoothing']:.2e}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Mejor var_smoothing : 1.00e-03
F1-macro CV         : 0.5027


Evaluación del mejor modelo sobre el conjunto de validación

In [16]:
best_clf = gs.best_estimator_
y_pred_best = best_clf.predict(X_val_ft)

evaluate(
    "nb_ft_v6", y_val, y_pred_best,
    hyperparams={**gs.best_params_, "vectorizer": "FastText cc.es.300"}
)


=== nb_ft_v2 ===
Hiperparámetros: {'var_smoothing': np.float64(0.001), 'vectorizer': 'FastText cc.es.300'}

F1-macro:  0.5019
Precision: 0.5155
Recall:    0.5050
Accuracy:  0.5380

              precision    recall  f1-score   support

    negativa     0.6792    0.4789    0.5617      4080
      neutra     0.2695    0.3397    0.3006      2040
    positiva     0.5979    0.6963    0.6433      4080

    accuracy                         0.5380     10200
   macro avg     0.5155    0.5050    0.5019     10200
weighted avg     0.5647    0.5380    0.5421     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      1954    1045      1081
neutra         517     693       830
positiva       406     833      2841


Reentrenar sobre `train + val` completo para el submission final. El clasificador se guarda solo

In [18]:
X_full_ft = np.vstack([X_train_ft, X_val_ft])
y_full    = np.concatenate([y_train, y_val])

best_clf.fit(X_full_ft, y_full)
save_model(best_clf, "nb_ft_v6")

# Generar predicciones y submission
preds = best_clf.predict(X_test_ft)
submission = pd.DataFrame({"id": test["id"], "label": preds})
submission.to_csv("submission_nb_ft_v6.csv", index=False)

print(f"Saved → submission_nb_ft_v6.csv  ({len(preds):,} predicciones)")
dist = {c: f"{(preds == c).mean():.1%}" for c in [0, 1, 2]}
print(f"Distribución: clase 0: {dist[0]} | clase 1: {dist[1]} | clase 2: {dist[2]}")

Modelo guardado → models/nb_ft_v6.joblib
Saved → submission_nb_ft_v6.csv  (8,500 predicciones)
Distribución: clase 0: 29.1% | clase 1: 24.5% | clase 2: 46.4%
